In [1]:

import json
import logging

import pandas as pd
import plotly.graph_objects as go
import torch

from kd_vs_hpo.common.analyze import (
    analyze_checkpoint_dir,
    group_confidence_interval,
    results_to_dataframe,
    summarize,
)
from kd_vs_hpo.common.analyze_new import analyze_dir, to_dataframe

logging.basicConfig(level=logging.WARNING)

%load_ext autoreload
%autoreload 2

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
def func(x):
    try:
        x = json.loads(x.params.kd.teachers_mapping.values[0])
    except:
        x = []
    x = sorted(int(i['idx']) for i in x)
    
    return ' '.join(map(str, x))

In [4]:

def get_results(path, device):
    results = analyze_checkpoint_dir(path, device=device)
    summary = summarize(results)
    df = results_to_dataframe(summary)
    df['teacher_names'] = df.apply(lambda x: func(x), axis=1)
    res = group_confidence_interval(df[df.role == 'student'].groupby(['arch_idx', 'teacher_names']).test_acc)
    return df, res


In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
results_noise = analyze_dir('experiments/checkpoints/kd_two_teachers_noise20', device=device)
results_base = analyze_dir('experiments/checkpoints/base_check_3_arch', device=device)
results_noise_base = analyze_dir('experiments/checkpoints/base_noise_20', device=device)
results_noise_base10 = analyze_dir('experiments/checkpoints/base_noise_10', device=device)
results_noise_base30 = analyze_dir('experiments/checkpoints/base_noise_30', device=device)
results_noise20_hpo = analyze_dir('experiments/checkpoints/kd_two_teachers_noise20_top_two_diff_hparams', device=device)

/home/ivanov.timofey51/kd_vs_hpo/.venv/lib/python3.11/site-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'teacher_ensemble' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['teacher_ensemble'])`.
/home/ivanov.timofey51/kd_vs_hpo/.venv/lib/python3.11/site-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'model' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['model'])`.
/home/ivanov.timofey51/kd_vs_hpo/.venv/lib/python3.11/site-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'criterion' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['criterion'])`.
/home/ivanov.timofey51/kd_vs_hpo/.venv/lib/python3.11/site-packages/lightning/pytorch/utilities/parsing

In [ ]:



res_noise = to_dataframe(results_noise)
res_base = to_dataframe(results_base)
res_noise_base = to_dataframe(results_noise_base)
res_noise_base10 = to_dataframe(results_noise_base10)
res_noise_base30 = to_dataframe(results_noise_base30)
res_noise20_hpo = to_dataframe(results_noise20_hpo)

In [8]:
res_noise[['teacher_test_accs', 'ensemble_test_acc']]

,teacher_test_accs,ensemble_test_acc
0,"[0.847100019454956, 0.8461999893188477]",0.8818
1,"[0.847100019454956, 0.8461999893188477]",0.8818
2,"[0.847100019454956, 0.8461999893188477]",0.8818
3,"[0.7077999711036682, 0.7067000269889832]",0.7333
4,"[0.7077999711036682, 0.7067000269889832]",0.7333
5,"[0.7077999711036682, 0.7067000269889832]",0.7333
6,"[0.7540000081062317, 0.746999979019165]",0.7906
7,"[0.7540000081062317, 0.746999979019165]",0.7906
8,"[0.7540000081062317, 0.746999979019165]",0.7906


In [9]:
res_noise20_hpo[['teacher_test_accs', 'ensemble_test_acc']]

,teacher_test_accs,ensemble_test_acc
0,"[0.8446000218391418, 0.8278999924659729]",0.8719
1,"[0.8446000218391418, 0.8278999924659729]",0.8719
2,"[0.8446000218391418, 0.8278999924659729]",0.8719
3,"[0.7013000249862671, 0.659500002861023]",0.7125
4,"[0.7013000249862671, 0.659500002861023]",0.7125
5,"[0.7013000249862671, 0.659500002861023]",0.7125
6,"[0.7652000188827515, 0.7071999907493591]",0.7767
7,"[0.7652000188827515, 0.7071999907493591]",0.7767
8,"[0.7652000188827515, 0.7071999907493591]",0.7767


In [ ]:
res_base = res_base[['student_arch_idx', 'student_seed', 'student_test_acc']].rename(columns={'student_test_acc': 'base_test_acc'})
res_base_noise = res_noise_base[['student_arch_idx', 'student_seed', 'student_test_acc']].rename(columns={'student_test_acc': 'base_noise_test_acc'})

In [24]:
res_hpo = res_noise20_hpo[['student_arch_idx', 'student_seed', 'student_test_acc','teacher_test_accs', 'ensemble_test_acc']]
res_hpo = res_hpo.rename(columns={x: x + '_hp' for x in res_hpo.columns if x not in ['student_arch_idx', 'student_seed']})

In [37]:
res_noise

,run_dir,seed,dataset,label_noise,student_test_acc,student_arch_idx,student_seed,teacher_test_accs,teacher_arch_idxs,teacher_seeds,...,params.kd.run_name,params.kd.optimizer_params.lr,params.kd.optimizer_params.momentum,params.kd.optimizer_params.weight_decay,params.kd.scheduler_params.T_max,params.kd.student,params.kd.teachers_mapping,params.kd.kd_loss._target_,params.kd.kd_loss.alpha,params.kd.kd_loss.temperature
0,experiments/checkpoints/kd_two_teachers_noise2...,17,cifar10,0.2,0.8818,11570,17,"[0.847100019454956, 0.8461999893188477]","[11570, 11570]","[84, 17]",...,,0.1,0.9,0.0005,200,11570,"[{'idx': 11570, 'path': '/home/ivanov.timofey5...",kd_vs_hpo.kd.losses.KullbackLeiblerKDLoss,0.5,1
1,experiments/checkpoints/kd_two_teachers_noise2...,42,cifar10,0.2,0.8868,11570,42,"[0.847100019454956, 0.8461999893188477]","[11570, 11570]","[84, 17]",...,,0.1,0.9,0.0005,200,11570,"[{'idx': 11570, 'path': '/home/ivanov.timofey5...",kd_vs_hpo.kd.losses.KullbackLeiblerKDLoss,0.5,1
2,experiments/checkpoints/kd_two_teachers_noise2...,84,cifar10,0.2,0.8827,11570,84,"[0.847100019454956, 0.8461999893188477]","[11570, 11570]","[84, 17]",...,,0.1,0.9,0.0005,200,11570,"[{'idx': 11570, 'path': '/home/ivanov.timofey5...",kd_vs_hpo.kd.losses.KullbackLeiblerKDLoss,0.5,1
3,experiments/checkpoints/kd_two_teachers_noise2...,17,cifar10,0.2,0.7294,1342,17,"[0.7077999711036682, 0.7067000269889832]","[1342, 1342]","[84, 17]",...,,0.1,0.9,0.0005,200,1342,"[{'idx': 1342, 'path': '/home/ivanov.timofey51...",kd_vs_hpo.kd.losses.KullbackLeiblerKDLoss,0.5,1
4,experiments/checkpoints/kd_two_teachers_noise2...,42,cifar10,0.2,0.7317,1342,42,"[0.7077999711036682, 0.7067000269889832]","[1342, 1342]","[84, 17]",...,,0.1,0.9,0.0005,200,1342,"[{'idx': 1342, 'path': '/home/ivanov.timofey51...",kd_vs_hpo.kd.losses.KullbackLeiblerKDLoss,0.5,1
5,experiments/checkpoints/kd_two_teachers_noise2...,84,cifar10,0.2,0.7260,1342,84,"[0.7077999711036682, 0.7067000269889832]","[1342, 1342]","[84, 17]",...,,0.1,0.9,0.0005,200,1342,"[{'idx': 1342, 'path': '/home/ivanov.timofey51...",kd_vs_hpo.kd.losses.KullbackLeiblerKDLoss,0.5,1
6,experiments/checkpoints/kd_two_teachers_noise2...,17,cifar10,0.2,0.7866,8712,17,"[0.7540000081062317, 0.746999979019165]","[8712, 8712]","[17, 84]",...,,0.1,0.9,0.0005,200,8712,"[{'idx': 8712, 'path': '/home/ivanov.timofey51...",kd_vs_hpo.kd.losses.KullbackLeiblerKDLoss,0.5,1
7,experiments/checkpoints/kd_two_teachers_noise2...,42,cifar10,0.2,0.7988,8712,42,"[0.7540000081062317, 0.746999979019165]","[8712, 8712]","[17, 84]",...,,0.1,0.9,0.0005,200,8712,"[{'idx': 8712, 'path': '/home/ivanov.timofey51...",kd_vs_hpo.kd.losses.KullbackLeiblerKDLoss,0.5,1
8,experiments/checkpoints/kd_two_teachers_noise2...,84,cifar10,0.2,0.7888,8712,84,"[0.7540000081062317, 0.746999979019165]","[8712, 8712]","[17, 84]",...,,0.1,0.9,0.0005,200,8712,"[{'idx': 8712, 'path': '/home/ivanov.timofey51...",kd_vs_hpo.kd.losses.KullbackLeiblerKDLoss,0.5,1


In [36]:
res_hpo

,student_arch_idx,student_seed,student_test_acc_hp,teacher_test_accs_hp,ensemble_test_acc_hp
0,11570,17,0.8801,"[0.8446000218391418, 0.8278999924659729]",0.8719
1,11570,42,0.8749,"[0.8446000218391418, 0.8278999924659729]",0.8719
2,11570,84,0.8742,"[0.8446000218391418, 0.8278999924659729]",0.8719
3,1342,17,0.7200,"[0.7013000249862671, 0.659500002861023]",0.7125
4,1342,42,0.7136,"[0.7013000249862671, 0.659500002861023]",0.7125
5,1342,84,0.7231,"[0.7013000249862671, 0.659500002861023]",0.7125
6,8712,17,0.7964,"[0.7652000188827515, 0.7071999907493591]",0.7767
7,8712,42,0.7902,"[0.7652000188827515, 0.7071999907493591]",0.7767
8,8712,84,0.7941,"[0.7652000188827515, 0.7071999907493591]",0.7767


In [26]:
result = res_noise.merge(res_base).merge(res_base_noise).merge(res_hpo)

In [65]:
result.spent_flops

0    4298215680000000
1    4298215680000000
2    4298215680000000
3    1106714880000000
4    1106714880000000
5    1106714880000000
6    1106714880000000
7    1106714880000000
8    1106714880000000
Name: spent_flops, dtype: int64

In [30]:
conf_student = group_confidence_interval(result.groupby(['student_arch_idx']).student_test_acc).reset_index()
conf_student['type'] = 'student_noise'
conf_base = group_confidence_interval(res_base.groupby(['student_arch_idx']).base_test_acc).reset_index()
conf_base['type'] = 'base'
conf_base_noise = group_confidence_interval(res_base_noise.groupby(['student_arch_idx']).base_noise_test_acc).reset_index()
conf_base_noise['type'] = 'base_noise'
conf_student_hp = group_confidence_interval(res_hpo.groupby(['student_arch_idx']).student_test_acc_hp).reset_index()
conf_student_hp['type'] = 'student_noise_hp'

In [43]:
conf_student_hp_ext.drop_duplicates()

,student_arch_idx,n,mean,std,sem,ci_low,ci_high,min,max,type,ensemble_test_acc_hp
0,1342,3,0.718900,0.004845,0.002797,0.706865,0.730935,0.7136,0.7231,student_noise_hp,0.7125
3,8712,3,0.793567,0.003134,0.001810,0.785781,0.801353,0.7902,0.7964,student_noise_hp,0.7767
6,11570,3,0.876400,0.003223,0.001861,0.868393,0.884407,0.8742,0.8801,student_noise_hp,0.8719


In [47]:
conf_student_ext = conf_student.merge(result[['student_arch_idx', 'ensemble_test_acc']], on='student_arch_idx')
conf_student_hp_ext = conf_student_hp.merge(result[['student_arch_idx', 'ensemble_test_acc_hp']], on='student_arch_idx')
ci = pd.concat([conf_student_hp_ext, conf_student_ext, conf_base, conf_base_noise], axis=0).drop_duplicates()
ci[['student_arch_idx', 'mean', 'std', 'ci_low', 'ci_high', 'min', 'max', 'type', 'ensemble_test_acc', 'ensemble_test_acc_hp']]

,student_arch_idx,mean,std,ci_low,ci_high,min,max,type,ensemble_test_acc,ensemble_test_acc_hp
0,1342,0.718900,0.004845,0.706865,0.730935,0.7136,0.7231,student_noise_hp,NaN,0.7125
3,8712,0.793567,0.003134,0.785781,0.801353,0.7902,0.7964,student_noise_hp,NaN,0.7767
6,11570,0.876400,0.003223,0.868393,0.884407,0.8742,0.8801,student_noise_hp,NaN,0.8719
0,1342,0.729033,0.002868,0.721910,0.736157,0.7260,0.7317,student_noise,0.7333,NaN
3,8712,0.791400,0.006502,0.775247,0.807553,0.7866,0.7988,student_noise,0.7906,NaN
6,11570,0.883767,0.002665,0.877146,0.890387,0.8818,0.8868,student_noise,0.8818,NaN
0,1342,0.737800,0.004557,0.726479,0.749121,0.7326,0.7411,base,NaN,NaN
1,8712,0.805433,0.009419,0.782035,0.828832,0.7996,0.8163,base,NaN,NaN
2,11570,0.913833,0.002040,0.908765,0.918902,0.9116,0.9156,base,NaN,NaN
0,1342,0.700400,0.011877,0.670895,0.729905,0.6867,0.7078,base_noise,NaN,NaN


In [34]:
result[['student_arch_idx', 'ensemble_test_acc', 'ensemble_test_acc_hp']]

,student_arch_idx,ensemble_test_acc,ensemble_test_acc_hp
0,11570,0.8818,0.8719
1,11570,0.8818,0.8719
2,11570,0.8818,0.8719
3,1342,0.7333,0.7125
4,1342,0.7333,0.7125
5,1342,0.7333,0.7125
6,8712,0.7906,0.7767
7,8712,0.7906,0.7767
8,8712,0.7906,0.7767


In [ ]:

def plot_ci(df):
    df.columns = [c.strip() for c in df.columns]
    df = df.drop(columns=[c for c in df.columns if c.startswith("Unnamed")])
    for c in ["student_arch_idx", "mean", "std", "ci_low", "ci_high", "min", "max"]:
        df[c] = df[c].astype(float)
    df["student_arch_idx"] = df["student_arch_idx"].astype(int)
    df["label"] = df["student_arch_idx"].astype(str) + " | " + df["type"]
    df["err_low"] = df["mean"] - df["ci_low"]
    df["err_high"] = df["ci_high"] - df["mean"]

    palette = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd",
            "#8c564b", "#e377c2", "#7f7f7f", "#bcbd22", "#17becf"]
    arch_colors = {arch: palette[i % len(palette)]
                for i, arch in enumerate(df["student_arch_idx"].unique())}

    fig = go.Figure()
    for arch in df["student_arch_idx"].unique():
        sub = df[df["student_arch_idx"] == arch]
        fig.add_trace(go.Scatter(
            y=sub["label"],
            x=sub["mean"],
            mode="markers",
            name=str(arch),
            marker=dict(color=arch_colors[arch], size=8),
            error_x=dict(
                type="data",
                symmetric=False,
                array=sub["err_high"],
                arrayminus=sub["err_low"],
                thickness=1.5,
                width=4,
                color=arch_colors[arch],
            ),
            customdata=sub[["std", "ci_low", "ci_high", "min", "max"]],
            hovertemplate=(
                "<b>%{y}</b><br>"
                "mean: %{x:.4f}<br>"
                "CI: [%{customdata[1]:.4f}, %{customdata[2]:.4f}]<br>"
                "std: %{customdata[0]:.4f}<br>"
                "min/max: [%{customdata[3]:.4f}, %{customdata[4]:.4f}]<extra></extra>"
            ),
            showlegend=True,
        ))

        fig.add_trace(go.Scatter(  # min dots
            y=sub["label"],
            x=sub["min"],
            mode="markers",
            name="min",
            marker=dict(color=arch_colors[arch], size=5, symbol="diamond", opacity=0.9),
            customdata=sub[["type"]],
            legendgroup=f"min_{arch}",
            showlegend=False,
            hovertemplate="<b>%{y}</b><br>min: %{x:.4f}<extra></extra>",
        ))
        fig.add_trace(go.Scatter(  # max dots
            y=sub["label"],
            x=sub["max"],
            mode="markers",
            name="max",
            marker=dict(color=arch_colors[arch], size=5, symbol="diamond", opacity=0.9),
            customdata=sub[["type"]],
            legendgroup=f"max_{arch}",
            showlegend=False,
            hovertemplate="<b>%{y}</b><br>max: %{x:.4f}<extra></extra>",
        ))
        fig.add_trace(go.Scatter(  # ensemble
            y=sub["label"],
            x=sub["ensemble_test_acc"],
            mode="markers",
            name="ensemble acc",
            marker=dict(color=arch_colors[arch], size=10, symbol="star-open", opacity=0.9),
            customdata=sub[["type"]],
            legendgroup=f"ensemble_{arch}",
            showlegend=False,
            hovertemplate="<b>%{y}</b><br>ensemble: %{x:.4f}<extra></extra>",
        ))
        fig.add_trace(go.Scatter(  # ensemble hp
            y=sub["label"],
            x=sub["ensemble_test_acc_hp"],
            mode="markers",
            name="ensemble hp acc",
            marker=dict(color=arch_colors[arch], size=10, symbol="star-open", opacity=0.9),
            customdata=sub[["type"]],
            legendgroup=f"ensemble_hp_{arch}",
            showlegend=False,
            hovertemplate="<b>%{y}</b><br>ensemble hp: %{x:.4f}<extra></extra>",
        ))

    EXPERIMENT = 'base_noise: CI for test accuracy for training single arch on noisy data;\nbase: CI for test accuracy for training single arch on clean data;\nstudent_noise: CI for test accuracy for training student with teacher on noisy data'

    fig.update_layout(
        title="Confidence intervals for distillation with ensemble of 3x teachers 11570 (strong), 20% noise. CIFAR-10",
        xaxis_title="Value",
        yaxis_title="student_arch_idx | type",
        template="plotly_white",
        annotations=[dict(
            text=EXPERIMENT,
            xref="paper", yref="paper",
            x=0, y=1.05, xanchor="left", yanchor="bottom",
            showarrow=False, font=dict(size=10, color="#555555"),
        )],
    )
    return fig

In [12]:
def plot_ci_vs_noise(noise_dfs, base_noise_dfs=None, base_acc=None, title="Knowledge distillation depending on label noise", facet=False):
    """Plot mean student accuracy +/- CI vs label noise, coloured by student arch idx.

    noise_dfs: iterable of (noise_rate, df) pairs, where df is the output of
    to_dataframe(analyze_dir(...)) for a KD run dir at that noise rate and has
    the columns 'student_arch_idx', 'student_test_acc' and 'ensemble_test_acc'.

    base_noise_dfs: optional iterable of (noise_rate, df) pairs for solo (non-KD)
    models trained on the same noise levels, e.g. base_check_3_arch for 0% and
    base_noise_10/20/30 for the noisy runs. Each arch is drawn as a dashed line
    with its own confidence band in a darker shade of the arch colour.

    base_acc: optional dict mapping arch idx -> base (non-KD) accuracy; drawn as
    dashed horizontal lines in a slightly darker shade of each arch colour.

    facet: if True, put every arch on its own subplot (one arch per panel) instead
    of overlaying them; each arch keeps its own palette colour as in the main plot.
    """
    def _shade(hex_color, factor=0.55):
        hex_color = hex_color.lstrip("#")
        r, g, b = (int(hex_color[i:i + 2], 16) for i in (0, 2, 4))
        return f"#{int(r * factor):02x}{int(g * factor):02x}{int(b * factor):02x}"

    def _rgba(hex_color, alpha=0.5):
        hex_color = hex_color.lstrip("#")
        r, g, b = (int(hex_color[i:i + 2], 16) for i in (0, 2, 4))
        return f"rgba({r}, {g}, {b}, {alpha})"

    def _ci_frame(noise_dfs):
        frames = []
        for noise, df in noise_dfs:
            conf = group_confidence_interval(
                df.groupby("student_arch_idx").student_test_acc
            ).reset_index()
            conf["noise"] = noise
            frames.append(conf)
        if not frames:
            return pd.DataFrame()
        out = pd.concat(frames, axis=0).drop_duplicates().sort_values("noise")
        out["err_low"] = out["mean"] - out["ci_low"]
        out["err_high"] = out["ci_high"] - out["mean"]
        return out

    ci_frames, ens_frames = [], []
    for noise, df in noise_dfs:
        conf = group_confidence_interval(
            df.groupby("student_arch_idx").student_test_acc
        ).reset_index()
        conf["noise"] = noise
        ci_frames.append(conf)

        ens = df[["student_arch_idx", "ensemble_test_acc"]].dropna()
        if not ens.empty:
            ens = ens.groupby("student_arch_idx").ensemble_test_acc.mean().reset_index()
            ens = ens.rename(columns={"student_arch_idx": "arch", "ensemble_test_acc": "ensemble_acc"})
            ens["noise"] = noise
            ens_frames.append(ens)

    df = pd.concat(ci_frames, axis=0).drop_duplicates().sort_values("noise")
    df["err_low"] = df["mean"] - df["ci_low"]
    df["err_high"] = df["ci_high"] - df["mean"]
    ens_df = pd.concat(ens_frames, axis=0) if ens_frames else pd.DataFrame(columns=["arch", "noise", "ensemble_acc"])
    base_df = _ci_frame(base_noise_dfs) if base_noise_dfs else pd.DataFrame()

    palette = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd",
            "#8c564b", "#e377c2", "#7f7f7f", "#bcbd22", "#17becf"]
    archs = sorted(set(df["student_arch_idx"].unique()) | set(ens_df["arch"].unique()))
    if not base_df.empty:
        archs = sorted(set(archs) | set(base_df["student_arch_idx"].unique()))
    if base_acc:
        archs = sorted(set(archs) | set(base_acc.keys()))
    arch_colors = {arch: palette[i % len(palette)] for i, arch in enumerate(archs)}

    if facet:
        from plotly.subplots import make_subplots
        fig = make_subplots(
            rows=len(archs), cols=1,
            subplot_titles=[f"arch {int(a)}" for a in archs],
            shared_xaxes=True, vertical_spacing=0.08,
        )
    else:
        fig = go.Figure()

    def _add_arch(arch, row=None, col=None):
        color = arch_colors[arch]
        legend = not facet
        where = {} if row is None else {"row": row, "col": col}
        sub = df[df["student_arch_idx"] == arch].sort_values("noise")
        if not sub.empty:
            fig.add_trace(go.Scatter(  # CI upper bound
                x=sub["noise"], y=sub["ci_high"],
                mode="lines", line=dict(width=0, color=color),
                showlegend=False, hoverinfo="skip",
            ), **where)
            fig.add_trace(go.Scatter(  # CI band (lower bound filled towards upper bound)
                x=sub["noise"], y=sub["ci_low"],
                mode="lines", line=dict(width=0, color=color),
                fill="tonexty", fillcolor=_rgba(color, 0.5),
                showlegend=False, hoverinfo="skip",
            ), **where)
            sub = sub.copy()
            sub["arch"] = int(arch)
            fig.add_trace(go.Scatter(  # mean line + markers
                x=sub["noise"], y=sub["mean"],
                mode="lines+markers", name=f"KD student {int(arch)}",
                line=dict(color=color, width=2),
                marker=dict(color=color, size=8),
                error_y=dict(
                    type="data", symmetric=False,
                    array=sub["err_high"], arrayminus=sub["err_low"],
                    thickness=1.5, width=4, color=color,
                ),
                customdata=sub[["arch", "std", "ci_low", "ci_high", "min", "max"]],
                showlegend=legend,
                hovertemplate=(
                    "<b>arch %{customdata[0]}</b><br>"
                    "noise: %{x}%<br>"
                    "mean: %{y:.4f}<br>"
                    "CI: [%{customdata[2]:.4f}, %{customdata[3]:.4f}]<br>"
                    "std: %{customdata[1]:.4f}<br>"
                    "min/max: [%{customdata[4]:.4f}, %{customdata[5]:.4f}]<extra></extra>"
                ),
            ), **where)

        ens_sub = ens_df[ens_df["arch"] == arch]
        if not ens_sub.empty:
            fig.add_trace(go.Scatter(  # ensemble accuracy: stars connected by a dotted line
                x=ens_sub["noise"], y=ens_sub["ensemble_acc"],
                mode="lines+markers", name=f"Ensemble of {ens_sub.arch.iloc[0].item()}",
                line=dict(color=color, width=1.5, dash="dot"),
                marker=dict(color=color, size=9, symbol="x-dot", opacity=0.7),
                customdata=ens_sub[["arch", "noise"]],
                showlegend=legend,
                hovertemplate=(
                    "<b>arch %{customdata[0]}</b><br>"
                    "noise: %{customdata[1]}%<br>"
                    "ensemble: %{y:.4f}<extra></extra>"
                ),
            ), **where)

        base_sub = base_df[base_df["student_arch_idx"] == arch].sort_values("noise") if not base_df.empty else pd.DataFrame()
        if not base_sub.empty:
            base_color = _shade(color)
            fig.add_trace(go.Scatter(  # base CI upper bound
                x=base_sub["noise"], y=base_sub["ci_high"],
                mode="lines", line=dict(width=0, color=base_color),
                showlegend=False, hoverinfo="skip",
            ), **where)
            fig.add_trace(go.Scatter(  # base CI band
                x=base_sub["noise"], y=base_sub["ci_low"],
                mode="lines", line=dict(width=0, color=base_color),
                fill="tonexty", fillcolor=_rgba(base_color, 0.25),
                showlegend=False, hoverinfo="skip",
            ), **where)
            base_sub = base_sub.copy()
            base_sub["arch"] = int(arch)
            fig.add_trace(go.Scatter(  # solo model mean +/- CI
                x=base_sub["noise"], y=base_sub["mean"],
                mode="lines+markers", name=f"Simple training {int(arch)}",
                line=dict(color=base_color, width=1.5, dash="dash"),
                marker=dict(color=base_color, size=6, symbol="circle-open"),
                error_y=dict(
                    type="data", symmetric=False,
                    array=base_sub["err_high"], arrayminus=base_sub["err_low"],
                    thickness=1.2, width=3, color=base_color,
                ),
                customdata=base_sub[["arch", "std", "ci_low", "ci_high", "min", "max"]],
                showlegend=legend,
                hovertemplate=(
                    "<b>arch %{customdata[0]} (solo)</b><br>"
                    "noise: %{x}%<br>"
                    "mean: %{y:.4f}<br>"
                    "CI: [%{customdata[2]:.4f}, %{customdata[3]:.4f}]<br>"
                    "std: %{customdata[1]:.4f}<br>"
                    "min/max: [%{customdata[4]:.4f}, %{customdata[5]:.4f}]<extra></extra>"
                ),
            ), **where)

        if base_acc and arch in base_acc:
            fig.add_trace(go.Scatter(  # base accuracy dashed horizontal line
                x=[0, 30], y=[base_acc[arch], base_acc[arch]],
                mode="lines", name=f"Simple training {int(arch)} on clean data",
                line=dict(color=_shade(color), width=1.5, dash="dash"),
                showlegend=legend, hoverinfo="skip",
            ), **where)

    if facet:
        for row, arch in enumerate(archs, start=1):
            _add_arch(arch, row=row, col=1)
        fig.update_xaxes(tickmode="array", tickvals=[0, 10, 20, 30],
                ticktext=["0%", "10%", "20%", "30%"])
        fig.update_yaxes(title_text="Accuracy on test set", row=1, col=1)
    else:
        for arch in archs:
            _add_arch(arch)
        fig.update_layout(
            xaxis_title="Label noise (%)",
            yaxis_title="Accuracy on test set",
            xaxis=dict(tickmode="array", tickvals=[0, 10, 20, 30],
                    ticktext=["0%", "10%", "20%", "30%"]),
        )

    fig.update_layout(title=title, template="plotly_white", width=1200, height=1000, title_x=0.5)
    return fig


In [3]:
noise_dfs = [
    (0, to_dataframe(analyze_dir('experiments/checkpoints/kd_two_teachers_noise_0', device=device))),
    (10, to_dataframe(analyze_dir('experiments/checkpoints/kd_two_teachers_noise10', device=device))),
    (20, to_dataframe(analyze_dir('experiments/checkpoints/kd_two_teachers_noise20', device=device))),
    (30, to_dataframe(analyze_dir('experiments/checkpoints/kd_two_teachers_noise30', device=device))),
]

base_noise_dfs = [
    (0, to_dataframe(analyze_dir('experiments/checkpoints/base_check_3_arch', device=device))),
    (10, to_dataframe(analyze_dir('experiments/checkpoints/base_noise_10', device=device))),
    (20, to_dataframe(analyze_dir('experiments/checkpoints/base_noise_20', device=device))),
    (30, to_dataframe(analyze_dir('experiments/checkpoints/base_noise_30', device=device))),
]


/home/ivanov.timofey51/kd_vs_hpo/.venv/lib/python3.11/site-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'model' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['model'])`.
/home/ivanov.timofey51/kd_vs_hpo/.venv/lib/python3.11/site-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'criterion' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['criterion'])`.
/home/ivanov.timofey51/kd_vs_hpo/.venv/lib/python3.11/site-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'teacher_ensemble' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['teacher_ensemble'])`.
/home/ivanov.timofey51/kd_vs_hpo/.venv/lib/python3.11/site-packages/lightning/pytorch/utilities/parsing

In [50]:
base_df = to_dataframe(analyze_dir('experiments/checkpoints/base_check_3_arch', device=device))
base_acc = base_df.groupby('student_arch_idx').student_test_acc.mean().round(4).to_dict()

/home/ivanov.timofey51/kd_vs_hpo/.venv/lib/python3.11/site-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'model' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['model'])`.
/home/ivanov.timofey51/kd_vs_hpo/.venv/lib/python3.11/site-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'criterion' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['criterion'])`.
/home/ivanov.timofey51/kd_vs_hpo/.venv/lib/python3.11/site-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'kd_loss' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['kd_loss'])`.
/home/ivanov.timofey51/kd_vs_hpo/.venv/lib/python3.11/site-packages/lightning/pytorch/utilities/parsing.py:213: Attribute

In [61]:
plot_ci(ci) # add vertical line for ensemble

In [13]:
plot_ci_vs_noise(noise_dfs, base_noise_dfs=base_noise_dfs, title="Knowledge distillation depending on label noise", facet=False)


# base architectures

In [4]:
base_res = get_results('experiments/checkpoints/base_check_3_arch', device=device)

/home/ivanov.timofey51/kd_vs_hpo/.venv/lib/python3.11/site-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'model' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['model'])`.
/home/ivanov.timofey51/kd_vs_hpo/.venv/lib/python3.11/site-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'criterion' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['criterion'])`.
/home/ivanov.timofey51/kd_vs_hpo/.venv/lib/python3.11/site-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'kd_loss' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['kd_loss'])`.


In [5]:
base_res

,,n,mean,std,sem,ci_low,ci_high,min,max
arch_idx,teacher_names,,,,,,,,
1342,,3,0.737800,0.004557,0.002631,0.726479,0.749121,0.7326,0.7411
8712,,3,0.805433,0.009419,0.005438,0.782035,0.828832,0.7996,0.8163
11570,,3,0.913833,0.002040,0.001178,0.908765,0.918902,0.9116,0.9156


# kd 1 teacher different seeds

In [6]:
results_kd_1 = get_results('experiments/checkpoints/kd_one_teacher_different_seeds', device=device)

/home/ivanov.timofey51/kd_vs_hpo/.venv/lib/python3.11/site-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'model' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['model'])`.
/home/ivanov.timofey51/kd_vs_hpo/.venv/lib/python3.11/site-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'criterion' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['criterion'])`.
/home/ivanov.timofey51/kd_vs_hpo/.venv/lib/python3.11/site-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'teacher_ensemble' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['teacher_ensemble'])`.
/home/ivanov.timofey51/kd_vs_hpo/.venv/lib/python3.11/site-packages/lightning/pytorch/utilities/parsing

In [7]:
results_kd_1

n      mean       std       sem    ci_low   ci_high  \
arch_idx teacher_names                                                        
1342     11570          3  0.741233  0.004727  0.002729  0.729491  0.752976   
         1342           3  0.750500  0.004397  0.002538  0.739578  0.761422   
8712     11570          3  0.807700  0.007712  0.004452  0.788543  0.826857   
         1342           3  0.803233  0.005169  0.002985  0.790392  0.816075   
11570    11570          3  0.918067  0.002303  0.001330  0.912346  0.923787   
         1342           3  0.875367  0.002854  0.001648  0.868278  0.882456   

                           min     max  
arch_idx teacher_names                  
1342     11570          0.7358  0.7444  
         1342           0.7456  0.7541  
8712     11570          0.8014  0.8163  
         1342           0.8001  0.8092  
11570    11570          0.9157  0.9203  
         1342           0.8732  0.8786

# kd 2 teachers different seeds

In [8]:
results_kd_2 = get_results('experiments/checkpoints/kd_two_teachers_different_seeds', device=device)

/home/ivanov.timofey51/kd_vs_hpo/.venv/lib/python3.11/site-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'model' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['model'])`.
/home/ivanov.timofey51/kd_vs_hpo/.venv/lib/python3.11/site-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'criterion' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['criterion'])`.
/home/ivanov.timofey51/kd_vs_hpo/.venv/lib/python3.11/site-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'teacher_ensemble' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['teacher_ensemble'])`.
/home/ivanov.timofey51/kd_vs_hpo/.venv/lib/python3.11/site-packages/lightning/pytorch/utilities/parsing

In [9]:
results_kd_2

n      mean       std       sem    ci_low   ci_high  \
arch_idx teacher_names                                                        
8712     11570 11570    3  0.810833  0.008796  0.005078  0.788984  0.832683   
         1342 11570     9  0.810989  0.006782  0.002261  0.805775  0.816202   
         1342 1342      3  0.810767  0.002371  0.001369  0.804876  0.816657   

                           min     max  
arch_idx teacher_names                  
8712     11570 11570    0.8026  0.8201  
         1342 11570     0.8020  0.8226  
         1342 1342      0.8086  0.8133

# five hpo teachers

In [10]:
results_5_hpo = get_results('experiments/checkpoints/five_teachers', device=device)

/home/ivanov.timofey51/kd_vs_hpo/.venv/lib/python3.11/site-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'model' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['model'])`.
/home/ivanov.timofey51/kd_vs_hpo/.venv/lib/python3.11/site-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'criterion' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['criterion'])`.
/home/ivanov.timofey51/kd_vs_hpo/.venv/lib/python3.11/site-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'kd_loss' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['kd_loss'])`.


In [11]:
results_5_hpo

,,n,mean,std,sem,ci_low,ci_high,min,max
arch_idx,teacher_names,,,,,,,,
1342,,5,0.56644,0.148073,0.066220,0.382583,0.750297,0.4235,0.7423
11570,,5,0.66088,0.227112,0.101568,0.378883,0.942877,0.4031,0.9094


In [12]:
results_kd_1_hpo = get_results('checkpoints/kd_one_teacher_different_hparams', device=device)

/home/ivanov.timofey51/kd_vs_hpo/.venv/lib/python3.11/site-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'model' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['model'])`.
/home/ivanov.timofey51/kd_vs_hpo/.venv/lib/python3.11/site-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'criterion' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['criterion'])`.
/home/ivanov.timofey51/kd_vs_hpo/.venv/lib/python3.11/site-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'teacher_ensemble' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['teacher_ensemble'])`.
/home/ivanov.timofey51/kd_vs_hpo/.venv/lib/python3.11/site-packages/lightning/pytorch/utilities/parsing

In [13]:
results_kd_2_hpo = get_results('checkpoints/kd_two_teacher_different_hparams', device=device)

/home/ivanov.timofey51/kd_vs_hpo/.venv/lib/python3.11/site-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'model' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['model'])`.
/home/ivanov.timofey51/kd_vs_hpo/.venv/lib/python3.11/site-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'criterion' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['criterion'])`.
/home/ivanov.timofey51/kd_vs_hpo/.venv/lib/python3.11/site-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'teacher_ensemble' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['teacher_ensemble'])`.
/home/ivanov.timofey51/kd_vs_hpo/.venv/lib/python3.11/site-packages/lightning/pytorch/utilities/parsing

# high temp

In [4]:
results_kd_1_hpo_htemp = get_results('checkpoints/kd_one_teacher_different_hparams_high_temp', device=device)

/home/ivanov.timofey51/kd_vs_hpo/.venv/lib/python3.11/site-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'model' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['model'])`.
/home/ivanov.timofey51/kd_vs_hpo/.venv/lib/python3.11/site-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'criterion' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['criterion'])`.
/home/ivanov.timofey51/kd_vs_hpo/.venv/lib/python3.11/site-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'teacher_ensemble' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['teacher_ensemble'])`.
/home/ivanov.timofey51/kd_vs_hpo/.venv/lib/python3.11/site-packages/lightning/pytorch/utilities/parsing

# base noise 50%

In [5]:
base_noise = get_results('experiments/checkpoints/base_noise_50', device=device)

/home/ivanov.timofey51/kd_vs_hpo/.venv/lib/python3.11/site-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'model' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['model'])`.
/home/ivanov.timofey51/kd_vs_hpo/.venv/lib/python3.11/site-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'criterion' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['criterion'])`.
/home/ivanov.timofey51/kd_vs_hpo/.venv/lib/python3.11/site-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'kd_loss' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['kd_loss'])`.


# base noise 20%

In [4]:
base_noise20 = get_results('experiments/checkpoints/base_noise_20', device=device)

/home/ivanov.timofey51/kd_vs_hpo/.venv/lib/python3.11/site-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'model' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['model'])`.
/home/ivanov.timofey51/kd_vs_hpo/.venv/lib/python3.11/site-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'criterion' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['criterion'])`.
/home/ivanov.timofey51/kd_vs_hpo/.venv/lib/python3.11/site-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'kd_loss' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['kd_loss'])`.


# kd noise 50% 1 teacher

In [6]:
results_kd_1_noise = get_results('experiments/checkpoints/kd_one_teacher_different_seeds_noise_50', device=device)

/home/ivanov.timofey51/kd_vs_hpo/.venv/lib/python3.11/site-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'model' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['model'])`.
/home/ivanov.timofey51/kd_vs_hpo/.venv/lib/python3.11/site-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'criterion' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['criterion'])`.
/home/ivanov.timofey51/kd_vs_hpo/.venv/lib/python3.11/site-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'teacher_ensemble' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['teacher_ensemble'])`.
/home/ivanov.timofey51/kd_vs_hpo/.venv/lib/python3.11/site-packages/lightning/pytorch/utilities/parsing

# comparison

In [15]:
base_res

,,n,mean,std,sem,ci_low,ci_high,min,max
arch_idx,teacher_names,,,,,,,,
1342,,3,0.737800,0.004557,0.002631,0.726479,0.749121,0.7326,0.7411
8712,,3,0.805433,0.009419,0.005438,0.782035,0.828832,0.7996,0.8163
11570,,3,0.913833,0.002040,0.001178,0.908765,0.918902,0.9116,0.9156


In [16]:
results_kd_1

n      mean       std       sem    ci_low   ci_high  \
arch_idx teacher_names                                                        
1342     11570          3  0.741233  0.004727  0.002729  0.729491  0.752976   
         1342           3  0.750500  0.004397  0.002538  0.739578  0.761422   
8712     11570          3  0.807700  0.007712  0.004452  0.788543  0.826857   
         1342           3  0.803233  0.005169  0.002985  0.790392  0.816075   
11570    11570          3  0.918067  0.002303  0.001330  0.912346  0.923787   
         1342           3  0.875367  0.002854  0.001648  0.868278  0.882456   

                           min     max  
arch_idx teacher_names                  
1342     11570          0.7358  0.7444  
         1342           0.7456  0.7541  
8712     11570          0.8014  0.8163  
         1342           0.8001  0.8092  
11570    11570          0.9157  0.9203  
         1342           0.8732  0.8786

In [17]:
results_kd_1_hpo

n      mean       std       sem    ci_low   ci_high  \
arch_idx teacher_names                                                        
1342     11570          3  0.734033  0.012788  0.007383  0.702265  0.765801   
         1342           3  0.729467  0.026339  0.015207  0.664038  0.794895   
8712     11570          3  0.799167  0.001650  0.000953  0.795067  0.803266   
         1342           3  0.800533  0.009717  0.005610  0.776396  0.824671   
11570    11570          3  0.907667  0.014829  0.008561  0.870830  0.944503   
         1342           3  0.879133  0.011808  0.006817  0.849801  0.908465   

                           min     max  
arch_idx teacher_names                  
1342     11570          0.7203  0.7456  
         1342           0.6998  0.7501  
8712     11570          0.7975  0.8008  
         1342           0.7918  0.8110  
11570    11570          0.8906  0.9174  
         1342           0.8690  0.8921

In [ ]:
results_kd_1_hpo_htemp

n      mean       std       sem    ci_low   ci_high  \
arch_idx teacher_names                                                        
1342     11570          3  0.747233  0.019442  0.011225  0.698936  0.795531   
         1342           3  0.727167  0.020841  0.012032  0.675396  0.778938   
8712     11570          3  0.816167  0.016600  0.009584  0.774929  0.857404   
         1342           3  0.787533  0.014775  0.008530  0.750830  0.824237   
11570    11570          3  0.910867  0.010563  0.006099  0.884626  0.937107   
         1342           3  0.851300  0.043108  0.024888  0.744215  0.958385   

                           min     max  
arch_idx teacher_names                  
1342     11570          0.7248  0.7592  
         1342           0.7045  0.7455  
8712     11570          0.7995  0.8327  
         1342           0.7715  0.8006  
11570    11570          0.8987  0.9177  
         1342           0.8207  0.9006

In [18]:
results_kd_2

n      mean       std       sem    ci_low   ci_high  \
arch_idx teacher_names                                                        
8712     11570 11570    3  0.810833  0.008796  0.005078  0.788984  0.832683   
         1342 11570     9  0.810989  0.006782  0.002261  0.805775  0.816202   
         1342 1342      3  0.810767  0.002371  0.001369  0.804876  0.816657   

                           min     max  
arch_idx teacher_names                  
8712     11570 11570    0.8026  0.8201  
         1342 11570     0.8020  0.8226  
         1342 1342      0.8086  0.8133

In [19]:
results_kd_2_hpo

n      mean       std       sem    ci_low   ci_high  \
arch_idx teacher_names                                                        
8712     11570 11570    3  0.805700  0.003031  0.001750  0.798169  0.813231   
         1342 11570     9  0.803133  0.011400  0.003800  0.794371  0.811896   
         1342 1342      3  0.797733  0.011433  0.006601  0.769331  0.826136   

                           min     max  
arch_idx teacher_names                  
8712     11570 11570    0.8022  0.8075  
         1342 11570     0.7763  0.8153  
         1342 1342      0.7891  0.8107

In [7]:
base_noise

,,n,mean,std,sem,ci_low,ci_high,min,max
arch_idx,teacher_names,,,,,,,,
1342,,3,0.315000,0.048180,0.027817,0.195315,0.434685,0.2653,0.3615
8712,,3,0.289667,0.029663,0.017126,0.215978,0.363355,0.2648,0.3225
11570,,3,0.328133,0.064103,0.037010,0.168894,0.487373,0.2547,0.3729


In [5]:
base_noise20

,,n,mean,std,sem,ci_low,ci_high,min,max
arch_idx,teacher_names,,,,,,,,
1342,,3,0.700400,0.011877,0.006857,0.670895,0.729905,0.6867,0.7078
8712,,3,0.747733,0.005934,0.003426,0.732992,0.762474,0.7422,0.7540
11570,,3,0.845433,0.002155,0.001244,0.840080,0.850786,0.8430,0.8471


In [1]:
results_kd_1_noise

NameError: name 'results_kd_1_noise' is not defined

todo:
 - Paired Bootstrap for Comparing Teacher vs. Student 
 - добавить качество ансамбля самих учителей
 - шумные метки: всё повторяем - учим на ушмном датасете с нуля
 - cifar-100: по сидам
 - температура??
 - проверить что архитектура учителя влияет 
 - add vertical line for ensemble 
 - 10-20-30 each arch learns from themself
 - old exps with 3 teachers??
 - exp with different hyperparams on the noise
 - NAS посчитать со стартовыми архиитектурами initial_population_paths по одной из 
 - cifar100 по сидам
 - 